# Telegram Expense Tracker — Demo evaluada

Notebook principal de la tarea final. Se ejecuta de principio a fin y en orden, sin Telegram.
Cada sección corresponde a un criterio de la rúbrica o a un bono declarado (ver `docs/bonos.md`).

| Sección | Contenido | Estado |
|---|---|---|
| 0 | Setup y ficha de reproducción | **Disponible** |
| 1 | Caso de uso y criterio de éxito | **Disponible** |
| 2 | LLM y prompts | **Disponible** (llamadas reales: requieren `.env`) |
| 3 | ReAct | Pendiente |
| 4 | Historial | Pendiente |
| 5 | Seguridad | Pendiente |
| 6 | Router (bono) | Pendiente |
| 7 | Memoria avanzada (bono) | Pendiente |
| 8 | Herramienta de acción (bono) | **Disponible (parcial)**: Drive (Etapa 4); Sheets en la Etapa 5 |
| 9 | Juez LLM (bono) | Pendiente |
| 10 | Golden set (bono) | Pendiente |

## Sección 0 — Setup y ficha de reproducción

Esta sección prepara el entorno y no requiere `.env` ni red: solo muestra si cada variable está **definida** o **falta** (nunca su valor) y demuestra el trazador con un secreto falso que se enmascara. El modelo exacto se completa en la Etapa 3.

In [ ]:
import sys
from pathlib import Path

# El notebook vive en notebooks/: se agrega la raíz del repositorio al path.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Python:", sys.version.split()[0])
print("Raíz del repositorio:", ROOT.name)

In [ ]:
from app.config import config_status

# Solo presencia de variables, nunca valores.
for name, state in config_status().items():
    print(f"{name:<34} {state}")

### Demostración del trazador

Se registra un evento con un secreto **falso** construido en tiempo de ejecución. La consola y el archivo JSONL solo muestran `***`.

In [ ]:
import tempfile

from app.trace import Tracer

fake_key = "AIza" + "x" * 35  # secreto de ejemplo, no real
with tempfile.TemporaryDirectory() as tmp:
    tracer = Tracer(session="demo", trace_dir=tmp)
    tracer.record("USER_INPUT", {"texto": "Registra este recibo"})
    tracer.record("TOOL_CALL", {"tool": "analizar_recibo", "api_key": fake_key,
                                "nota": f"clave usada: {fake_key}"})
    contenido = tracer.path.read_text(encoding="utf-8")

print("Eventos:", len(tracer.events))
print("El secreto aparece en el JSONL:", fake_key in contenido)

## Sección 1 — Caso de uso y criterio de éxito

**Usuario:** persona que registra sus gastos personales y usa Telegram a diario.
**Entrada:** imagen de un recibo (JPG/PNG) y texto opcional.
**Salida:** archivo en Drive, fila nueva en Sheets y confirmación con el número de fila.

**Criterio observable:** dada una imagen válida de `data/receipts/`, el caso es exitoso cuando
(1) fecha, comercio y monto coinciden con `data/receipts/expected.json`;
(2) la categoría es una de las permitidas;
(3) Drive devuelve un `file_id` y un `web_view_link` que existen;
(4) la planilla tiene exactamente una fila más, con ese enlace;
(5) la respuesta contiene el número de fila.

**Por qué un LLM:** los recibos no tienen estructura fija y el flujo exige decidir entre registrar, pedir confirmación, pedir otra foto o rechazar.
**Por qué herramientas:** el LLM no puede persistir; Drive y Sheets devuelven observaciones verificables.

La celda siguiente muestra el documento completo del caso de uso.

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
display(Markdown((ROOT / "docs" / "use_case.md").read_text(encoding="utf-8")))

## Sección 2 — LLM y prompts

Criterio de rúbrica: **LLM real y trazabilidad** (modelo, configuración, prompts y traza de cada llamada).

- Modelo de ejecución confirmado por el autor: `gemini-3.5-flash-lite` (SDK oficial `google-genai`). El código **no** lo escribe: lo lee de `LLM_MODEL`.
- Todas las llamadas incluyen `SECURITY_SCOPE_v1`; el texto de las imágenes es **dato, no instrucción**.
- La última celda llama al LLM solo si hay `GEMINI_API_KEY` y `LLM_MODEL`; sin ellas se omite con un aviso y el notebook sigue corriendo.

In [ ]:
from app.config import config_status
from app.llm import EXTRACTION_TEMPERATURE

status = config_status()  # solo presencia, nunca valores
print(f"{'Variable':<34} Estado")
for name in ("GEMINI_API_KEY", "LLM_MODEL", "LLM_MAX_RETRIES", "LLM_MIN_SECONDS_BETWEEN_CALLS"):
    print(f"{name:<34} {status[name]}")
print()
print("ID de modelo confirmado por el autor: gemini-3.5-flash-lite (LLM_MODEL debe tenerlo)")
print("Fuentes: https://ai.google.dev/gemini-api/docs/models/gemini-3.5-flash-lite")
print("         https://ai.google.dev/gemini-api/docs/pricing")
print(f"Temperatura de extracción (EXTRACTION_TEMPERATURE): {EXTRACTION_TEMPERATURE}")
print("  0.0 según el prompt maestro; pendiente de verificación real")
print("  (Google recomienda 1.0 en Gemini 3: https://ai.google.dev/gemini-api/docs/gemini-3)")

### Prompts versionados

Todos los prompts viven en `app/prompts.py`. La traza registra el identificador, no el texto completo.

In [ ]:
from app.prompts import PROMPTS, compose_system_instruction

print("Identificadores de prompt:", ", ".join(PROMPTS))
print()
print(compose_system_instruction("ANALYZER_PROMPT_v1"))

### Recibos sintéticos

Imágenes generadas con `scripts/generate_receipts.py` (comercios ficticios, sin datos personales). Ver `data/README.md`.

In [ ]:
import json
from IPython.display import Image, display

RECEIPTS = ROOT / "data" / "receipts"
expected = json.loads((RECEIPTS / "expected.json").read_text(encoding="utf-8"))
for name, exp in expected.items():
    print(f"{name}: esperado {exp}")
    display(Image(filename=str(RECEIPTS / name), width=360))

### Llamada real a `analizar_recibo`

Se ejecuta solo si `GEMINI_API_KEY` y `LLM_MODEL` están definidas. Cada recibo consume una llamada de la capa gratuita; la traza muestra el modelo, los parámetros, los tokens y los reintentos ante 429.

In [ ]:
from app.llm import LLMCallError, LLMClient
from app.tools.analyzer import analizar_recibo
from app.trace import Tracer

if status["GEMINI_API_KEY"] == "definida" and status["LLM_MODEL"] == "definida":
    tracer = Tracer(session="notebook-seccion-2")
    llm = LLMClient(tracer=tracer)
    for name, exp in expected.items():
        print(f"\n== {name} (esperado: fecha={exp['fecha']}, monto={exp['monto']}) ==")
        try:
            print("Resultado:", analizar_recibo(RECEIPTS / name, llm=llm).model_dump())
        except LLMCallError as error:
            print("La API falló tras los reintentos:", error)
    print("\nContadores de la sesión:", llm.stats.as_dict())
else:
    print("omitido: falta GEMINI_API_KEY/LLM_MODEL (configura .env; ver .env.example).")
    print("Las secciones anteriores y las pruebas offline no los necesitan.")

## Sección 3 — Loop ReAct

Criterio de rúbrica: **ReAct integrado** (1,0). El LLM pide una herramienta pertinente, la observación vuelve al LLM y este decide repetir o responder, con una condición de parada explícita.

**Cómo funciona (`app/agent.py`):**
1. Se registra `USER_INPUT`. El LLM recibe el texto y solo un `image_id` de la imagen adjunta; nunca ve los bytes.
2. El LLM decide con *function calling* nativo (modo `AUTO`). El código **no fija el orden** `analizar → guardar → registrar`: lo decide el LLM.
3. Si pide una tool: se registra `TOOL_CALL`, el código la despacha, se registra `TOOL_RESULT` y la observación vuelve al LLM como `function_response`. El contenido del modelo se reenvía sin modificar (conserva las firmas de pensamiento de Gemini 3).
4. Si responde sin pedir tools, termina.

**Condiciones de parada** (evento `STOP` con su motivo):
- `respuesta_final`: el LLM responde sin tool call.
- `max_steps`: se alcanzan `MAX_STEPS = 6` decisiones sin respuesta final; la tool pedida en esa decisión **no se ejecuta** y se entrega una respuesta segura.
- `error_llm` / `respuesta_vacia`: la API falla tras los reintentos o el modelo no devuelve nada.

**Rieles en código** (valen aunque el LLM ignore el prompt): tool desconocida o argumentos faltantes devuelven un error como observación; `registrar_gasto` se bloquea si la URL no vino de `guardar_recibo` en esta ejecución, o si la extracción tiene confianza < 0,7 o algún campo `desconocido`; `guardar_recibo` exige un análisis previo.

**Degradación controlada:** con solo la clave de Gemini, `analizar_recibo` corre de verdad; si faltan las credenciales de Google, `guardar_recibo` y `registrar_gasto` devuelven un error estructurado que el LLM recibe como observación y responde con honestidad. Nunca se simula un éxito de Drive o Sheets.

In [ ]:
from app.agent import AGENT_PROMPT_ID, MAX_STEPS, build_tool_declarations
from app.llm import AGENT_TEMPERATURE
from app.models import CONFIDENCE_THRESHOLD

print(f"MAX_STEPS = {MAX_STEPS} | umbral de confianza = {CONFIDENCE_THRESHOLD}")
print(f"Prompt del agente: {AGENT_PROMPT_ID} (+ SECURITY_SCOPE_v1) | AGENT_TEMPERATURE = {AGENT_TEMPERATURE}")
print("Modo de function calling: AUTO | ejecución automática del SDK: desactivada")
print("\nTools expuestas al LLM:")
for declaration in build_tool_declarations()[0].function_declarations:
    params = declaration.parameters_json_schema
    print(f"- {declaration.name}({', '.join(params['properties'])})")
    print(f"    {declaration.description}")

### Ejecución real del agente

Se ejecuta solo si `GEMINI_API_KEY` y `LLM_MODEL` están definidas; si no, se omite. Con las credenciales de Google completas, el agente guarda el recibo en Drive y escribe **una** fila en la planilla de prueba (si el gasto ya existe, informa el duplicado). Sin ellas, el flujo degrada con honestidad. La traza se imprime en vivo y se guarda en `traces/notebook-seccion-3.jsonl`.

In [ ]:
from app.agent import ExpenseAgent
from app.config import config_status, load_settings
from app.google_auth import token_exists
from app.llm import LLMCallError, LLMClient
from app.models import EventType
from app.trace import Tracer

status = config_status()
if status["GEMINI_API_KEY"] == "definida" and status["LLM_MODEL"] == "definida":
    google_ok = (
        status["DRIVE_FOLDER_ID"] == "definida"
        and status["SHEET_ID"] == "definida"
        and token_exists(load_settings())
    )
    print("Google configurado:", "sí" if google_ok else "no (degradación honesta)", "\n")
    tracer = Tracer(session="notebook-seccion-3")
    llm = LLMClient(tracer=tracer)
    try:
        result = ExpenseAgent(llm=llm, tracer=tracer).run(
            "Registra este recibo", RECEIPTS / "receipt_normal.jpg"
        )
    except LLMCallError as error:
        print("La API falló tras los reintentos:", error)
    else:
        print("\n== RESUMEN ==")
        print("Secuencia de eventos:", " -> ".join(e.event_type.value for e in result.events))
        print("Secuencia de tools:  ", " -> ".join(result.tool_sequence) or "(ninguna)")
        print(f"Motivo de parada:     {result.stop_reason} ({result.steps} decisiones de {MAX_STEPS})")
        print("Respuesta final:     ", result.final_text)
        print("Contadores de la sesión:", llm.stats.as_dict())
else:
    print("omitido: falta GEMINI_API_KEY/LLM_MODEL (configura .env; ver .env.example).")
    print("Las pruebas offline de tests/test_stage6_agent.py cubren el loop con un LLM guionado.")

## Sección 8 — Herramienta de acción (Drive/Sheets)

Criterio del bono: **herramienta de acción** que modifica un sistema externo de forma segura. En esta etapa: `guardar_recibo` sube la imagen a **una carpeta de prueba** de Drive (`DRIVE_FOLDER_ID`) con OAuth de usuario y el scope mínimo `drive.file`. `registrar_gasto` (Sheets) agrega una fila de gasto en **una planilla de prueba** (`SHEET_ID`): valida antes de escribir, solo agrega filas (nunca borra ni sobrescribe) y no duplica si se repite la llamada.

- El `web_view_link` sale **siempre** de la respuesta de la API; nunca se construye a mano.
- Sin credenciales, la tool devuelve `success=False` con un mensaje claro y no lanza excepciones.
- Configuración paso a paso: `docs/setup_google.md`.

*Nota: esta sección se agrega al final por ahora; el orden definitivo de las secciones se fija en la Etapa 14.*

### Normalización del nombre del archivo (offline)

In [ ]:
from app.tools.drive import build_file_name

casos = [
    ("Jumbo", "2026-09-30"),
    ("Café Ñandú & Cía.", "2026-01-05"),
    ("desconocido", "desconocido"),
    ("../../etc/passwd", "2026-09-30"),
]
for comercio, fecha in casos:
    print(f"{comercio!r:<24} {fecha:<12} -> {build_file_name(comercio, fecha)}")

### Subida real a Drive

Se ejecuta solo si `DRIVE_FOLDER_ID` está definida y existe el token OAuth (`scripts/google_auth.py`). Si no, se omite con un aviso y el notebook sigue corriendo.

In [ ]:
from app.config import config_status, load_settings
from app.google_auth import build_drive_service, token_exists
from app.tools.drive import guardar_recibo, verify_file_exists
from app.trace import Tracer

settings = load_settings()
faltan = []
if config_status()["DRIVE_FOLDER_ID"] == "falta":
    faltan.append("DRIVE_FOLDER_ID")
if not token_exists(settings):
    faltan.append("token OAuth (scripts/google_auth.py)")

if faltan:
    print("omitido: falta " + " y ".join(faltan) + ". Ver docs/setup_google.md.")
else:
    tracer = Tracer(session="notebook-seccion-8")
    service = build_drive_service(settings)
    resultado = guardar_recibo(
        RECEIPTS / "receipt_normal.jpg", "Comercio de Prueba", "2026-09-30",
        service=service, tracer=tracer, settings=settings,
    )
    print("Resultado:", resultado.model_dump())
    if resultado.success:
        meta = verify_file_exists(resultado.file_id, service)
        print("files.get confirma el archivo:", meta is not None and meta["id"] == resultado.file_id)

### Validación de `registrar_gasto` (offline, sin API)

La validación es un riel de **código**, no una instrucción del prompt: ocurre antes de cualquier llamada a Google. Aquí se prueban una categoría inválida, un monto no positivo y una URL que no es de Drive. En los tres casos `success=False` y no se hace ninguna llamada de red.

In [ ]:
from app.tools.sheets import registrar_gasto

URL_DRIVE = "https://drive.google.com/file/d/EJEMPLO/view"
base = dict(fecha="2026-09-12", comercio="Los Aromos", monto=18490, categoria="Supermercado", recibo_url=URL_DRIVE)
rechazos = {
    "categoría inválida": {"categoria": "Cripto"},
    "monto no positivo": {"monto": -5},
    "monto desconocido": {"monto": "desconocido"},
    "URL que no es de Drive": {"recibo_url": "https://evil.example.com/file/d/1/view"},
    "URL http": {"recibo_url": "http://drive.google.com/file/d/1/view"},
}
for nombre, cambio in rechazos.items():
    r = registrar_gasto(**{**base, **cambio})  # sin service: la validación falla antes de tocar la API
    print(f"{nombre:<26} success={r.success}  {r.error}")

### Evidencia del bono: estado ANTES, llamada, estado DESPUÉS y repetición

Se ejecuta solo si `DRIVE_FOLDER_ID` y `SHEET_ID` están definidas y existe el token OAuth. Si no, se omite con un aviso. Pasos:

1. **ANTES:** conteo de filas de datos y última fila de la planilla de prueba.
2. **Llamada:** sube el recibo a Drive y registra el gasto con el `web_view_link` devuelto por la API.
3. **DESPUÉS:** el conteo sube en 1 y la última fila es la nueva (`row_number` sale de `updates.updatedRange`).
4. **Repetición:** la misma llamada devuelve `duplicate=True` y el conteo no cambia. Así se puede repetir con seguridad (solo append + deduplicación por fecha, comercio y monto).

El comercio lleva una marca de tiempo UTC para que cada ejecución del notebook use una fila nueva.

In [ ]:
from datetime import datetime, timezone

from app.config import config_status, load_settings
from app.google_auth import build_drive_service, build_sheets_service, token_exists
from app.tools.drive import guardar_recibo
from app.tools.sheets import get_sheet_snapshot, registrar_gasto
from app.trace import Tracer

settings = load_settings()
faltan = [n for n in ("DRIVE_FOLDER_ID", "SHEET_ID") if config_status()[n] == "falta"]
if not token_exists(settings):
    faltan.append("token OAuth (scripts/google_auth.py)")

if faltan:
    print("omitido: falta " + " y ".join(faltan) + ". Ver docs/setup_google.md.")
else:
    tracer = Tracer(session="notebook-seccion-8-sheets")
    drive = build_drive_service(settings)
    sheets = build_sheets_service(settings)
    esperado = json.loads((RECEIPTS / "expected.json").read_text(encoding="utf-8"))["receipt_normal.jpg"]
    comercio = f"{esperado['comercio']} (notebook {datetime.now(timezone.utc):%Y-%m-%d %H:%M:%SZ})"

    antes = get_sheet_snapshot(sheets, settings)
    print("ANTES   :", antes)

    subida = guardar_recibo(RECEIPTS / "receipt_normal.jpg", comercio, esperado["fecha"],
                            service=drive, tracer=tracer, settings=settings)
    print("Drive   :", subida.web_view_link if subida.success else subida.error)
    if subida.success:
        args = dict(fecha=esperado["fecha"], comercio=comercio, monto=esperado["monto"],
                    categoria=esperado["categoria"], recibo_url=subida.web_view_link,
                    service=sheets, tracer=tracer, settings=settings)
        resultado = registrar_gasto(**args)
        print("LLAMADA :", resultado.model_dump())
        despues = get_sheet_snapshot(sheets, settings)
        print("DESPUÉS :", despues)
        print("conteo +1:", despues["row_count"] == antes["row_count"] + 1)

        repetido = registrar_gasto(**args)
        final = get_sheet_snapshot(sheets, settings)
        print("REPETIR :", repetido.model_dump())
        print("duplicado y conteo sin cambio:", repetido.duplicate and final["row_count"] == despues["row_count"])